In [1]:
library(ArchR)
library(BSgenome.Hsapiens.UCSC.hg38)

library(Seurat)
library(SeuratDisk)
library(AUCell)
library(readxl)
library(patchwork)
library(dplyr)
library(data.table)
library(ggplot2)
library(reshape2)
library(GSEABase)
library(broom)
library(ggrepel)
library(gridExtra) # Or use the `patchwork` package for flexibility
library(ComplexHeatmap)
library(circlize)

suppressMessages(library(tidyverse))
suppressMessages(library(pROC))
#suppressMessages(library(bestglm))
suppressMessages(library(rpart))
suppressMessages(library(rpart.plot))
suppressMessages(library(randomForest))
suppressMessages(library(xgboost))
suppressMessages(library(FNN))
suppressMessages(library(e1071))


set.seed(1)
addArchRThreads(threads = 42) 
addArchRGenome("hg38")

proj_dir = '/Aging/'

### directories
data_dir = '/singleCell/Ruzicka_snRNA_Seq/data/'
data_dir2 = '/Aging/Data/Anderson/snRNA_Seq/'
out_dir2 = '/Aging/Aging_Clock/Output/'

PLOTDIR = '/Aging/Aging_Clock/Plots/'
setwd(PLOTDIR)


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

In [85]:
# Base metadata
metadata = read.csv('/Aging/Data/Anderson/Multiomics/Raw/GSE214979_cell_metadata.csv')

# Extract metadata from ATAC Project
proj <- loadArchRProject(path = "/Aging/Data/Anderson/Multiomics/ATAC/ArchR/ArchRProject_Neurons", force = FALSE, showLogo = FALSE)
metadata1 <- as.data.frame(getCellColData(proj))
metadata1 <- metadata1[1:15]
metadata1$X <- rownames(metadata1)
metadata1$X = gsub('GSE214979_atac_fragments#','',metadata1$X)

# ER_score
ER_score <- read.csv(paste0(data_dir2,"GSE214979_ER_score.csv"))
ER_score$X = gsub('GSE214979_atac_fragments#','',ER_score$X)

# AUCell
AUCell <- read.csv(paste0(data_dir2,"GSE214979_metadata_AUCell_all_scores.csv"), row.names = 1)
AUCell <- AUCell[c(5,27,60:91)]
AUCell$geneset0_status <- coalesce(AUCell$geneset_pos0.y, AUCell$geneset_neg0.y)
AUCell$geneset24_status <- coalesce(AUCell$geneset_pos24.y, AUCell$geneset_neg24.y)
q1 = round(quantile(AUCell$geneset_dd.x,0.25,na.rm=T),4)
q2 = round(quantile(AUCell$geneset_dd.x,0.75,na.rm=T),4)
AUCell = AUCell %>% mutate(geneset_dd_status = case_when(geneset_dd.x <= q1 ~ "low",
                                                 geneset_dd.x >= q2 ~ "high"))
#----------------------------------------------------------------

metadata <- left_join(metadata, metadata1, by='X')
metadata <- left_join(AUCell, metadata, by='X')
metadata <- left_join(metadata, ER_score, by='X') # 56215 x 84

# Remove Ex-NRGN from Analysis
metadata <- metadata[-which(metadata$predicted.id.x=='Ex-NRGN'),]

Successfully loaded ArchRProject!



In [98]:
# AD vs Ctrl

df <- metadata[metadata$Status == "Ctrl",]
df$Vulnerability_Status <- ifelse(
  df$predicted.id.x %in% c('Ex-L5b_VEN', 'Ex-L6b_SEMA3D', 'Ex−L6b_SEMA3E', 
                         'In−Rosehip_CHST9', 'In-Reelin', 'In-SST'),
  'Vulnerable',
  'Resilient'
)
df$Vulnerability_Status <- as.factor(df$Vulnerability_Status)
table(df$Vulnerability_Status)

set.seed(0)
# Balancing classes
df1 <- df[df$Vulnerability_Status == 'Resilient',]
df1<- df1[sample(nrow(df1), min(table(df$Vulnerability_Status))),]$X
df2 <- df[df$Vulnerability_Status == 'Vulnerable',]$X
barcodes <- c(df1,df2)
df <- df[df$X %in% barcodes,]

table(df$Vulnerability_Status)


df <- df[,c('Vulnerability_Status','geneset_sc0','geneset_sc24', 'geneset_dd.x',
                'can_sen.x','sen_init.x','sen_resp.x', 'Epigenetic_Erosion_Score')] #'Epigenetic_Erosion_Score'
#df$Vulnerability_Status <- ifelse(df$Vulnerability_Status == 'Vulnerable', 1, 0)

# Scaling
# KNN/SVM require the data to be scaled -- will use scaled data for all models to allow for comparison between model  
df <- na.omit(df)
df[,-which(names(df) %in% c('predicted.id.x', 'Vulnerability_Status'))] <- data.frame(scale(df[,-which(names(df)%in% c('predicted.id.x', 'Vulnerability_Status'))]))

### Data Splitting into 70% Train & 30% Test ###
set.seed(0)
s <- sample(nrow(df),round(0.7*nrow(df)))
df.train <- df[s,]
df.test  <- df[-s,]


 Resilient Vulnerable 
     24064       4016 


 Resilient Vulnerable 
      4016       4016 

In [99]:
# Manuscript Figure 8 ---------------------------------------------------------
#------------------------------------------------------------------------------

# Classification tree

pdf(paste0(PLOTDIR, 'Anderson_DT_classifiers_Ctrl_ER.pdf'), width = 5, height = 5)

rpart.out <- rpart(Vulnerability_Status~.,data=df.train, method = "class") # control = rpart.control(cp = 0.05)

# 1-SE rule: largest CP within 1 std error of min xerror
cp.table <- rpart.out$cptable
min.err <- min(cp.table[,"xerror"])
min.err.idx <- which.min(cp.table[,"xerror"])
cutoff <- min.err + cp.table[min.err.idx,"xstd"]
cp.1se <- max(cp.table[cp.table[,"xerror"] <= cutoff, "CP"])

rpart.pruned <- prune(rpart.out, cp = cp.1se)

resp.prob <- predict(rpart.pruned,newdata=df.test,type="prob")[,2] # class 1 prob
resp.pred <- predict(rpart.pruned,newdata=df.test, type='class')

# Mis-classification rate 
DT_MCR = round(mean(resp.pred!=df.test$Vulnerability_Status),2)
DT_MCR
# Confusion Matrix + Performance Metrics 
cf <- table(resp.pred, df.test$Vulnerability_Status)
cf
TN <- cf[1,1]
FP <- cf[2,1]
FN <- cf[1,2]
TP <- cf[2,2]
# Senstivity (TPR) / Recall 
DT_TPR = round(TP / (TP + FN),2)
# Specificity (TNR)
DT_TNR = round(TN / (TN + FP),2)
# Balanced accuracy
DT_BAc = round((DT_TPR + DT_TNR) / 2, 2)
# Accuracy 
DT_Ac = round((TP + TN) / (TP + TN + FP + FN),2)
# Precision 
DT_Pr = round(TP / (TP + FP),2)

# Plot Tree
rpart.plot(rpart.pruned, extra = 104, box.palette = c("#FFFFFF", "#21FE80"))

# ROC
DT_roc_obj <- roc(df.test$Vulnerability_Status, resp.prob)
DT_AUC <- round(DT_roc_obj$auc,2)
#DT_CI <- ci.auc(DT_roc_obj, method = "bootstrap", boot.n = 2000)
DT_CI <- ci.se(DT_roc_obj, specificities = seq(0, 1, 0.1), boot.n = 2000)
plot(DT_roc_obj, main = "ROC Curve for Decision Tree")
text(0.5, 0.5, paste("AUC =", round(DT_AUC, 2)), col = "red", adj = c(1, 0))
#Area under the curve: 72% chance that the model will be able to distinguish between Vulnerable and Resilient wine.

# Youden's J = Sensitivity+Specificity−1
J <- DT_roc_obj$sensitivities + DT_roc_obj$specificities - 1
w <- which.max(J)
threshold <- DT_roc_obj$thresholds[w]
class <- ifelse(resp.prob>threshold,'Vulnerable', 'Resilient')
# Mis-classification rate 
DT_MCR_J = round(mean(class!=df.test$Vulnerability_Status),2)
DT_MCR_J
# Confusion Matrix
table(class,df.test$Vulnerability_Status)

table1 = data.frame(Model='Decision Tree', AUC = DT_AUC, MCR_J = DT_MCR_J)
table1

#--------------------------------------------------------------

## Random Forest 

rf.out = randomForest(Vulnerability_Status~.,data=df.train,importance=TRUE)
resp.prob = predict(rf.out,newdata=df.test,type="prob")[,2] # class 1 prob
resp.pred <- ifelse(resp.prob>0.5,'Vulnerable', 'Resilient')
# Mis-classification rate 
RF_MCR = round(mean(resp.pred!=df.test$Vulnerability_Status),2)
RF_MCR
# Confusion Matrix + Performance metrics
cf = table(resp.pred,df.test$Vulnerability_Status)
cf
TN <- cf[1,1]
FP <- cf[2,1]
FN <- cf[1,2]
TP <- cf[2,2]
# Senstivity (TPR) / Recall 
RF_TPR = round(TP / (TP + FN),2)
# Specificity (TNR)
RF_TNR = round(TN / (TN + FP),2)
# Balanced accuracy
RF_BAc = round((RF_TPR + RF_TNR) / 2, 2)
# Accuracy 
RF_Ac = round((TP + TN) / (TP + TN + FP + FN),2)
# Precision 
RF_Pr = round(TP / (TP + FP),2)


# Variable importance plot
varImpPlot(rf.out,type=1)

# ROC
RF_roc_obj <- roc(df.test$Vulnerability_Status, resp.prob)
RF_AUC <- round(RF_roc_obj$auc,2)
RF_CI <- ci.se(RF_roc_obj, specificities = seq(0, 1, 0.1), boot.n = 2000)
plot(RF_roc_obj, main = "ROC Curve for Random Forest")
text(0.5, 0.5, paste("AUC =", round(RF_AUC, 2)), col = "red", adj = c(1, 0))
#Area under the curve: 89% chance that the model will be able to distinguish between Vulnerable and Resilient wine.

# Youden's J = Sensitivity+Specificity−1
J <- RF_roc_obj$sensitivities + RF_roc_obj$specificities - 1
w <- which.max(J)
rf_threshold <- RF_roc_obj$thresholds[w]
class <- ifelse(resp.prob>rf_threshold,'Vulnerable', 'Resilient')
# Mis-classification rate 
RF_MCR_J = round(mean(class!=df.test$Vulnerability_Status),2)
RF_MCR_J

table1 = data.frame(Model='Random Forest', AUC = RF_AUC, MCR_J = RF_MCR_J)
table1

#--------------------------------------------------------------

## XGBOOST

set.seed(1)
# Convert Vulnerable and Resilient to 1 and 0 for XGBoost
resp.train <- ifelse(df.train[, which(names(df.train) == 'Vulnerability_Status')] == 'Vulnerable', 1, 0)
resp.test <- ifelse(df.test[, which(names(df.test) == 'Vulnerability_Status')] == 'Vulnerable', 1, 0)
pred.train <- df.train[,-which(names(df.train)=='Vulnerability_Status')]
pred.test  <- df.test[,-which(names(df.test)=='Vulnerability_Status')]

train <- xgb.DMatrix(data=as.matrix(pred.train),label=resp.train)
test <- xgb.DMatrix(data=as.matrix(pred.test),label=resp.test)
xgb.cv.out <- xgb.cv(params=list(objective="binary:logistic"),train,nrounds=30,nfold=5,verbose=0, eval_metric = "error")
cat("The optimal number of trees is ",which.min(xgb.cv.out$evaluation_log$test_error_mean),"\n")

xgb.out <- xgboost(train,nrounds=which.min(xgb.cv.out$evaluation_log$test_error_mean),params=list(objective="binary:logistic"),verbose=0)
resp.prob <- predict(xgb.out,newdata=test) # class 1 prob
resp.pred <- ifelse(resp.prob>0.5,"1","0")
# Mis-classification rate 
XG_MCR = round(mean(resp.pred!=resp.test),2)
# Confusion Matrix + Performance Metrics
cf = table(resp.pred,resp.test)
cf
TN <- cf[1,1]
FP <- cf[2,1]
FN <- cf[1,2]
TP <- cf[2,2]
# Senstivity (TPR) / Recall 
XG_TPR = round(TP / (TP + FN),2)
# Specificity (TNR)
XG_TNR = round(TN / (TN + FP),2)
# Balanced accuracy
XG_BAc = round((XG_TPR + XG_TNR) / 2,2)
# Accuracy 
XG_Ac = round((TP + TN) / (TP + TN + FP + FN),2)
# Precision 
XG_Pr = round(TP / (TP + FP),2)

# ROC
XG_roc_obj <- roc(resp.test, resp.prob)
XG_AUC <- round(XG_roc_obj$auc,2)
XG_CI <- ci.se(XG_roc_obj, specificities = seq(0, 1, 0.1), boot.n = 2000)
plot(XG_roc_obj, main = "ROC Curve for XGBoost")
text(0.5, 0.5, paste("AUC =", round(XG_AUC, 2)), col = "red", adj = c(1, 0))
#Area under the curve: 86% chance that the model will be able to distinguish between Vulnerable and Resilient wine.

# Youden's J = Sensitivity+Specificity−1
J <- XG_roc_obj$sensitivities + XG_roc_obj$specificities - 1
w <- which.max(J)
threshold <- XG_roc_obj$thresholds[w]
class <- ifelse(resp.prob>threshold,"1","0")
# Mis-classification rate 
XG_MCR_J = round(mean(class!=resp.test),2)
XG_MCR_J
# Confusion Matrix
table(class,resp.test)

imp.out <- xgb.importance(model=xgb.out)
xgb.plot.importance(importance_matrix=imp.out,col="#000435")

table1 = data.frame(Model='XGBoost', AUC = XG_AUC, MCR_J = XG_MCR_J)
table1

#--------------------------------------------------------------

# Print table of models' metrics 


# Overlay of ROC curve 
plot(DT_roc_obj, col = "#98FF98")  # Light green
plot(DT_CI, type = "shape", col = rgb(152, 255, 152, 50, maxColorValue = 255), add = TRUE)  # Light green shaded
text(0.1, 0.4, paste("DT AUC =", round(DT_AUC, 2)), col = "#98FF98", adj = c(1, 0))

par(new = TRUE)
plot(RF_roc_obj, col = "#001F3F")  # Navy
plot(RF_CI, type = "shape", col = rgb(0, 31, 63, 50, maxColorValue = 255), add = TRUE)  # Navy shaded
text(0.1, 0.3, paste("RF AUC =", round(RF_AUC, 2)), col = "#001F3F", adj = c(1, 0))

par(new = TRUE)
plot(XG_roc_obj, col = "#FF6F61")  # Coral
plot(XG_CI, type = "shape", col = rgb(255, 111, 97, 50, maxColorValue = 255), add = TRUE)  # Coral shaded
text(0.1, 0.2, paste("XGBoost AUC =", round(XG_AUC, 2)), col = "#FF6F61", adj = c(1, 0))


# Assuming you have the tables for each model already created
# Combine the tables into one
results <- rbind(
  data.frame(Model='Decision Tree', AUC=DT_AUC, MCR=DT_MCR, MCR_J=DT_MCR_J, TPR=DT_TPR, TNR=DT_TNR , BAc=DT_BAc, Ac=DT_Ac, Pr=DT_Pr),
  data.frame(Model='Random Forest', AUC=RF_AUC, MCR=RF_MCR, MCR_J=RF_MCR_J, TPR=RF_TPR, TNR=RF_TNR , BAc=RF_BAc, Ac=RF_Ac, Pr=RF_Pr),
  data.frame(Model='XGBoost', AUC=XG_AUC, MCR=XG_MCR, MCR_J=XG_MCR_J, TPR=XG_TPR, TNR=XG_TNR , BAc=XG_BAc, Ac=XG_Ac, Pr=XG_Pr)
)

# Print the results table
print(results)
# Create a bar plot for mean classification error based on Youden's J
barplot(
  height = results$MCR,
  names.arg = results$Model,
  col = c("#98FF98", "#001F3F", "#FF6F61"),  # Custom colors
  main = "Mean Classification Error",
  ylab = "Mean Classification Error (MCR)",
  ylim = c(0, 1)  # Adjust based on your expected error range
)
# Adding the error values on top of the bars
text(x = seq_along(results$MCR), 
     y = results$MCR, 
     labels = round(results$MCR, 2), 
     pos = 3)

dev.off()

[1] 0.28

            
resp.pred    Resilient Vulnerable
  Resilient        904        410
  Vulnerable       208        699

Setting levels: control = Resilient, case = Vulnerable

Setting direction: controls < cases



[1] 0.28

            
class        Resilient Vulnerable
  Resilient        904        410
  Vulnerable       208        699

...

     resp.test
class   0   1
    0 834 218
    1 278 891

Model,AUC,MCR_J
<chr>,<dbl>,<dbl>
XGBoost,0.85,0.22


Warning message in plot.ci.se(DT_CI, type = "shape", col = rgb(152, 255, 152, 50, :
“Low definition shape.”
Warning message in plot.ci.se(RF_CI, type = "shape", col = rgb(0, 31, 63, 50, maxColorValue = 255), :
“Low definition shape.”
Warning message in plot.ci.se(XG_CI, type = "shape", col = rgb(255, 111, 97, 50, :
“Low definition shape.”


          Model  AUC  MCR MCR_J  TPR  TNR  BAc   Ac   Pr
1 Decision Tree 0.76 0.28  0.28 0.63 0.81 0.72 0.72 0.77
2 Random Forest 0.85 0.23  0.23 0.81 0.73 0.77 0.77 0.75
3       XGBoost 0.85 0.23  0.22 0.81 0.73 0.77 0.77 0.75


pdf 
  2